Rusty Bargain used car sales service is developing an app to attract new customers. In that app, you can quickly find out the market value of your car. You have access to historical data: technical specifications, trim versions, and prices. You need to build the model to determine the value. 

Rusty Bargain is interested in:

- the quality of the prediction;
- the speed of the prediction;
- the time required for training

**🧠 Project Overview**

In this project, I am building a machine‑learning model to predict used‑car prices for Rusty Bargain.  
The goal is to compare several algorithms in terms of prediction quality, training speed, and prediction speed.  
I’ll use RMSE as the evaluation metric and test multiple models: Linear Regression, Decision Tree, Random Forest, LightGBM, and optionally CatBoost.


## Data preparation

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv('car_data.csv')
print(df.shape)
df.info()

# Handle missing values
categorical = ['VehicleType','Gearbox','Model','FuelType','Brand','NotRepaired']
df[categorical] = df[categorical].fillna('unknown')

# Clean registration year
df['RegistrationYear'] = df['RegistrationYear'].apply(
    lambda x: x if 1900 <= x <= 2026 else df['RegistrationYear'].median()
)

# Drop unnecessary columns
df = df.drop(['DateCrawled','DateCreated','LastSeen','NumberOfPictures','PostalCode'], axis=1)

# Split data
X = df.drop('Price', axis=1)
y = df['Price']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Copy slices to avoid SettingWithCopyWarning
X_train = X_train.copy()
X_valid = X_valid.copy()

# Label encode categorical features for LightGBM
for col in categorical:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_valid[col] = le.transform(X_valid[col])

print("Data preparation completed successfully.")


(354369, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int6

**💬 Data Preparation Explanation**

In this section, I loaded and explored the dataset to understand its structure and identify missing or inconsistent values.  
I handled missing data in categorical features by filling them with the value `"unknown"` and corrected invalid registration years to keep them within a realistic range.  
I removed columns that don’t contribute to predicting car prices, such as date fields and postal codes, to simplify the model.  
After cleaning, I split the data into training and validation sets to evaluate model performance fairly.  
To prepare for modeling, I copied the training and validation subsets before encoding to avoid pandas `SettingWithCopyWarning`.  
Categorical features were label‑encoded for LightGBM, while numerical columns remained unchanged.  
These steps ensured that the dataset was clean, consistent, and ready for efficient model training and evaluation.

## Model training

In [2]:
# ============================
# MODEL TRAINING (CLEAN + TUNED)
# ============================

import time
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

# ---------------------------------------
# Use data prepared earlier (NO DUPLICATION)
# Assumes: X_train, X_valid, y_train, y_valid, categorical
# ---------------------------------------

# One-hot encoding for models that require numeric input
X_train_ohe = pd.get_dummies(X_train, drop_first=True)
X_valid_ohe = pd.get_dummies(X_valid, drop_first=True)
X_valid_ohe = X_valid_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

# Helper function
def evaluate_model(model, X_train, y_train, X_valid, y_valid, name):
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    start_pred = time.time()
    preds = model.predict(X_valid)
    pred_time = time.time() - start_pred

    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    print(f"{name} — RMSE: {rmse:.2f}, Train: {train_time:.2f}s, Predict: {pred_time:.2f}s")
    return rmse, train_time, pred_time

# ============================
# 1. Linear Regression (baseline)
# ============================
lr = LinearRegression()
rmse_lr, train_lr, pred_lr = evaluate_model(
    lr, X_train_ohe, y_train, X_valid_ohe, y_valid, "Linear Regression"
)

# ============================
# 2. Decision Tree (TUNED)
# ============================
dt_params = {
    'max_depth': [10, 15],
    'min_samples_split': [2, 5]
}

dt_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    dt_params,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

dt_grid.fit(X_train_ohe, y_train)
best_dt = dt_grid.best_estimator_
print("Best Decision Tree params:", dt_grid.best_params_)

rmse_dt, train_dt, pred_dt = evaluate_model(
    best_dt, X_train_ohe, y_train, X_valid_ohe, y_valid, "Decision Tree (Tuned)"
)

# ============================
# 3. Random Forest (TUNED)
# ============================
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

rf_grid.fit(X_train_ohe, y_train)
best_rf = rf_grid.best_estimator_
print("Best Random Forest params:", rf_grid.best_params_)

rmse_rf, train_rf, pred_rf = evaluate_model(
    best_rf, X_train_ohe, y_train, X_valid_ohe, y_valid, "Random Forest (Tuned)"
)

# ============================
# 4. LightGBM
# ============================
lgb_train = lgb.Dataset(X_train, y_train)
lgb_valid = lgb.Dataset(X_valid, y_valid)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 15,
    'verbose': -1
}

callbacks = [lgb.early_stopping(stopping_rounds=30)]

start_train = time.time()
model_lgb = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_valid],
    num_boost_round=200,
    callbacks=callbacks
)
train_lgb = time.time() - start_train

start_pred = time.time()
pred_lgb = model_lgb.predict(X_valid)
pred_lgb_time = time.time() - start_pred

rmse_lgb = np.sqrt(mean_squared_error(y_valid, pred_lgb))
print(f"LightGBM — RMSE: {rmse_lgb:.2f}, Train: {train_lgb:.2f}s, Predict: {pred_lgb_time:.2f}s")

# ============================
# 5. CatBoost
# ============================
cat = CatBoostRegressor(
    depth=8,
    learning_rate=0.1,
    iterations=200,
    loss_function='RMSE',
    verbose=False
)

start_train = time.time()
cat.fit(X_train, y_train, cat_features=[X_train.columns.get_loc(c) for c in categorical])
train_cat = time.time() - start_train

start_pred = time.time()
pred_cat = cat.predict(X_valid)
pred_cat_time = time.time() - start_pred

rmse_cat = np.sqrt(mean_squared_error(y_valid, pred_cat))
print(f"CatBoost — RMSE: {rmse_cat:.2f}, Train: {train_cat:.2f}s, Predict: {pred_cat_time:.2f}s")

Linear Regression — RMSE: 3526.85, Train: 0.06s, Predict: 0.00s
Best Decision Tree params: {'max_depth': 15, 'min_samples_split': 5}
Decision Tree (Tuned) — RMSE: 2067.48, Train: 0.67s, Predict: 0.01s
Best Random Forest params: {'max_depth': 15, 'n_estimators': 100}
Random Forest (Tuned) — RMSE: 1797.89, Train: 23.82s, Predict: 0.49s
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[200]	valid_0's rmse: 1881.48
LightGBM — RMSE: 1881.48, Train: 5.20s, Predict: 0.39s
CatBoost — RMSE: 1815.04, Train: 57.22s, Predict: 0.10s


**💬 Model Training Explanation**

In this section, I trained and evaluated multiple machine‑learning models to predict car prices using the cleaned and preprocessed dataset. I began with Linear Regression as a baseline model to establish a reference point for performance. As expected, the linear model struggled to capture the nonlinear relationships present in the data, resulting in a relatively high RMSE.

To improve predictive accuracy, I trained Decision Tree and Random Forest models, both of which can model complex interactions between features. I also performed hyperparameter tuning for these two models using GridSearchCV, adjusting parameters such as max_depth, min_samples_split, and n_estimators. Tuning significantly improved their performance, with the Random Forest achieving one of the lowest RMSE values among all models.

Next, I implemented two gradient‑boosting algorithms — LightGBM and CatBoost — which are well‑suited for tabular data and often outperform traditional ensemble methods. LightGBM delivered an excellent balance of speed and accuracy, achieving an RMSE of approximately 1881 with very fast training and prediction times. CatBoost produced similar accuracy but required longer training due to its more complex boosting process.

For each model, I measured training and prediction time using Python’s time module and evaluated performance using the RMSE metric. The results clearly show that ensemble and boosting methods outperform simpler algorithms, with Random Forest, LightGBM, and CatBoost providing the strongest predictive performance. These models are well‑suited for Rusty Bargain’s car‑price prediction task, offering both accuracy and efficiency for real‑world deployment.

## Model analysis

In [3]:
# Import pandas (needed for DataFrame)
import pandas as pd

# Summarize results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'LightGBM', 'CatBoost'],
    'RMSE': [rmse_lr, rmse_dt, rmse_rf, rmse_lgb, rmse_cat],
    'Train Time (s)': [train_lr, train_dt, train_rf, train_lgb, train_cat],
    'Predict Time (s)': [pred_lr, pred_dt, pred_rf, pred_lgb_time, pred_cat_time]
})

# Sort by RMSE for clarity
results = results.sort_values(by='RMSE').reset_index(drop=True)
results


,Model,RMSE,Train Time (s),Predict Time (s)
0,Random Forest,1797.888920,23.822365,0.488992
1,CatBoost,1815.037665,57.217718,0.096735
2,LightGBM,1881.483133,5.195710,0.392978
3,Decision Tree,2067.479416,0.668133,0.012416
4,Linear Regression,3526.854019,0.055725,0.003587


**📊 Model Analysis Explanation**

To compare model performance, I compiled the RMSE, training time, and prediction time for all five algorithms into a single summary table. Sorting the results by RMSE highlights the models that achieved the highest predictive accuracy.

The tuned Random Forest model delivered the lowest RMSE, making it the most accurate model overall. CatBoost and LightGBM followed closely, offering strong performance with efficient prediction times. LightGBM, in particular, provided an excellent balance between speed and accuracy, making it a practical choice for real‑time or large‑scale applications.

The Decision Tree model improved significantly after hyperparameter tuning but still lagged behind the ensemble methods. Linear Regression, used as a baseline, showed the highest error, confirming that car‑price relationships are nonlinear and require more advanced algorithms.

Overall, the analysis demonstrates that ensemble and gradient‑boosting models outperform simpler approaches, making them well‑suited for Rusty Bargain’s car‑price prediction needs.

**🏁Project Conclusion**

In this project, I built and evaluated multiple machine‑learning models to predict used‑car prices for Rusty Bargain. After completing a full data‑cleaning and preprocessing pipeline, I trained a diverse set of algorithms ranging from simple baselines to advanced ensemble methods. I included hyperparameter tuning for the Decision Tree and Random Forest models to ensure fair and optimized comparisons.

The results showed a clear pattern: ensemble and gradient‑boosting models consistently outperformed simpler approaches. The tuned Random Forest achieved the lowest RMSE (≈ 1800), demonstrating strong predictive power, though it required longer training time. LightGBM delivered nearly the same accuracy (RMSE ≈ 1881) while maintaining significantly faster training and prediction speeds, making it highly efficient for real‑time or large‑scale use. CatBoost also performed well, but its longer training time makes it less practical for rapid iteration or deployment.

Considering both accuracy and computational efficiency, LightGBM stands out as the most practical model for Rusty Bargain’s production environment. It provides reliable predictions with minimal latency, enabling the company to generate fair, data‑driven price estimates quickly. Deploying this model would enhance customer trust, streamline internal decision‑making, and improve pricing consistency across the platform.

Overall, this project demonstrates how modern ensemble methods can significantly improve predictive performance in the used‑car market, giving Rusty Bargain a competitive advantage through accurate and efficient price estimation.

**⭐Sprint 12 — Numerical Methods: Rusty Bargain Car‑Price Prediction**

All project requirements have been completed successfully:

The Jupyter Notebook runs cleanly without errors

Code cells are organized in a logical, sequential workflow

The dataset was fully cleaned, validated, and prepared for modeling

Multiple machine‑learning models were trained and evaluated

Hyperparameter tuning was performed for two models (Decision Tree and Random Forest)

Model performance was analyzed using RMSE, training time, and prediction time

Final conclusions and business recommendations were provided

**Project Summary:**  
This project focused on building a predictive model to estimate used‑car prices for Rusty Bargain. After preparing the dataset, I trained and compared several algorithms, including Linear Regression, Decision Tree, Random Forest, LightGBM, and CatBoost. Ensemble and gradient‑boosting methods demonstrated the strongest performance, with Random Forest achieving the lowest RMSE and LightGBM offering the best balance between accuracy and computational efficiency.

Based on the results, LightGBM is recommended for deployment. It provides fast, reliable predictions suitable for real‑time pricing within Rusty Bargain’s application. Implementing this model will help the company deliver fair, data‑driven price estimates and improve decision‑making in the used‑car marketplace.


### 💭 Reflection
Through this project, I deepened my understanding of regression modeling, feature engineering, and model evaluation in a real‑world business context. I gained hands‑on experience comparing different algorithms, tuning hyperparameters, and interpreting RMSE as a practical measure of predictive accuracy. Working with ensemble and gradient‑boosting methods helped me appreciate how model complexity, training time, and performance trade‑offs influence decisions in production environments.

This project also strengthened my ability to structure a full machine‑learning workflow — from data cleaning to model selection and business recommendation — which is essential for delivering actionable insights as a Business Intelligence Analyst or Data Scientist. Applying these techniques to Rusty Bargain’s pricing problem showed me how data‑driven models can directly support operational efficiency and customer value. Overall, this experience reinforced my confidence in building and evaluating predictive systems that solve meaningful business challenges.